# PhenoAgeSaoPaulo

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
5. [Load features](#Load-features)
6. [Load weights into base model](#Load-weights-into-base-model)
7. [Load reference values](#Load-reference-values)
8. [Load preprocess and postprocess objects](#Load-preprocess-and-postprocess-objects)
10. [Check all clock parameters](#Check-all-clock-parameters)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)
12. [Clear directory](#Clear-directory)

Let's first import some packages:

In [1]:
import os
import inspect
import shutil
import json
import math
import torch
import pandas as pd
import pyaging as pya

## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.PhenoAgeSaoPaulo)

class PhenoAgeSaoPaulo(pyagingModel):
    """PhenoAge refit without creatinine, albumin, and alkaline phosphatase."""

    def __init__(self):
        super().__init__()
        for name in ["m_n", "m_d", "ba_n", "ba_d", "ba_i"]:
            self.register_buffer(name, torch.empty(0))

    def preprocess(self, x):
        """Apply BioAge's log1p transform to C-reactive protein.

        Notes
        -----
        The refit's CRP coefficient is fit against ``log1p(CRP in mg/dL)``, not
        the natural log the published ``PhenoAge`` uses, so the two clocks read
        the same raw column differently.
        """
        return log1p_crp(self.features, x)

    def postprocess(self, x):
        """Convert the Gompertz mortality score to phenotypic age.

        Notes
        -----
        The constants are refit alongside the coefficients and differ from
        Levine's published ones, so they travel as buffers rather than being
        hardcoded the way ``PhenoAge.postprocess`` hardc

In [3]:
model = pya.models.PhenoAgeSaoPaulo()

## Define clock metadata

Each `# Paper:` comment reproduces the evidence recorded for that field in `clocks/metadata/evidence_ledger.jsonl`; `validate_metadata.py` compares the two, so they cannot drift apart. This clock is a **refit**, not a published model, so its provenance has two halves: the method and its citation come from Levine et al. (2018), while the coefficients and the mortality-to-age constants were produced locally by `clocks/extract_bioage_params.R` calling `BioAge::phenoage_calc()`. Fields describing the fit itself are therefore sourced from that code rather than from either paper.

In [4]:
model.metadata["clock_name"] = 'phenoagesaopaulo'
model.metadata["data_type"] = 'clinical biomarkers'  # Paper: a novel measure of 'phenotypic age' was developed using clinical data from the third National Health and Nutrition Examination Survey (NHANES)
model.metadata["species"] = 'Homo sapiens'  # Paper: our analytical sample included 9,926 adults with complete biomarker data
model.metadata["year"] = 2026
model.metadata["approved_by_author"] = '⌛'
model.metadata["citation"] = 'Levine, M. E., et al. "An epigenetic biomarker of aging for lifespan and healthspan." Aging 10.4 (2018): 573-591.'
model.metadata["doi"] = 'https://doi.org/10.18632/aging.101414'
model.metadata["notes"] = "PhenoAge refit on NHANES III adults aged 20-84 with BioAge::phenoage_calc(), dropping creatinine, albumin, and alkaline phosphatase and keeping the remaining six biomarkers plus chronological age. The coefficients and the mortality-to-age constants are refit, so they differ from the published PhenoAge and its constants do not apply. Biomarkers are on BioAge's SI-unit variants, which are natively pyaging's unit convention, and C-reactive protein is supplied raw in mg/dL and log1p-transformed inside the clock. The refit is pooled across sexes, so unlike kdmage and homeostaticdysregulation this clock takes no female column."
model.metadata["research_only"] = None
model.metadata["tissue"] = ['blood']  # Paper: measures like CRP, albumin, creatinine, glucose, etc.
model.metadata["predicts"] = ['phenotypic age']  # Paper: These nine biomarkers and chronological age were then combined in a phenotypic age estimate (in units of years)
model.metadata["training_target"] = ['mortality']  # Paper: gom = flexsurvreg(surv_form(bm_name), data = dat, dist = "gompertz")
model.metadata["unit"] = ['years']  # Paper: the mortality score was converted into units of years
model.metadata["model_type"] = 'Gompertz hazards regression with age calibration'  # Paper: These nine biomarkers and chronological age were then included in a parametric proportional hazards model based on the Gompertz distribution. Based on this model, we estimated the 10-year (120 months) mortality risk of the j-the individual. Next, the mortality score was converted into units of years
model.metadata["platform"] = ['clinical laboratory assays']  # Paper: forty-two clinical markers
model.metadata["population"] = 'adults'  # Paper: filter(age >= 20, age <= 84)
model.metadata["journal"] = 'Aging'  # Paper: Aging 10(4)
model.metadata["last_author"] = 'Steve Horvath'  # Paper: author list
model.metadata["n_features"] = 7  # Code: six biomarkers plus age
model.metadata["citations"] = 3594  # Paper: shared with phenoage/dnamphenoage (same DOI)
model.metadata["citations_date"] = '2026-07-05'

## Download clock dependencies

The fitted parameters were produced by `clocks/extract_bioage_params.R`, which runs [dayoonkwon/BioAge](https://github.com/dayoonkwon/BioAge)'s `phenoage_calc()` against NHANES III with `fit = NULL`, and are checked in under `clocks/bioage_params/phenoagesaopaulo.json`. The file carries the linear `coefficients` and `intercept`, the refit gompertz constants `m_n`, `m_d`, `ba_n`, `ba_d`, `ba_i`, and the `training_mean` / `training_n` of the rows the fit actually used.

In [5]:
with open("../bioage_params/phenoagesaopaulo.json") as handle:
    params = json.load(handle)

params["features"], params["training_n"]

(['glucose',
  'c_reactive_protein',
  'lymphocyte_percent',
  'mean_cell_volume',
  'red_cell_distribution_width',
  'white_blood_cell_count',
  'age'],
 14735)

## Load features

In [6]:
model.features = params["features"]
model.features

['glucose',
 'c_reactive_protein',
 'lymphocyte_percent',
 'mean_cell_volume',
 'red_cell_distribution_width',
 'white_blood_cell_count',
 'age']

#### Normal feature ranges

Every feature above is registered in `pyaging`'s feature range registry, which is the single source of truth for units and plausible bounds; the clock stores those units in `model.feature_units`, so a saved clock is self-describing and carries its own units even if the package registry later standardizes differently. `tests/test_clock_metadata.py` asserts every built clock's stored copy still matches the registry, so a registry correction cannot be silently shadowed by a stale one. `pya.utils.get_feature_ranges("phenoagesaopaulo")` reports them for a saved clock.

The refit was run on BioAge's SI-unit variants, so no post-hoc unit conversion is applied anywhere in this clock. As in `kdmage` and `homeostaticdysregulation`, `c_reactive_protein` is supplied **raw, in mg/dL**: BioAge's `lncrp` biomarker is `log1p(CRP in mg/dL)` — not the natural log that the published `phenoage` uses — so `PhenoAgeSaoPaulo.preprocess` applies `log1p` internally and the CRP coefficient stays on the `log1p` scale. A raw CRP is clamped to 0.01 mg/dL first, which is the registered lower bound, so a below-detection reading coded as `0` or an absent column cannot send `-inf` into the linear predictor.

There is no `female` feature. The refit pools the sexes into one gompertz fit, so there is no sex-specific parameter set to select and nothing for a `female` column to do.

In [ ]:
feature_ranges = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
model.feature_units = [record["unit"] for record in feature_ranges]
pd.DataFrame.from_records(feature_ranges)

## Load weights into base model

The clock is a plain linear predictor followed by a fixed two-stage transform, so the base model is a `LinearModel` and the gompertz constants ride along as buffers.

Those constants are the point of this clock. `phenoage_calc(fit = NULL)` fits `flexsurvreg(Surv(time, status) ~ biomarkers + age, dist = "gompertz")` for the mortality half and a second, age-only gompertz for the calibration half, deriving `m_n`, `m_d`, `ba_n`, `ba_d`, and `ba_i` from those two fits. They are therefore specific to *this* biomarker set and *this* training sample. Levine's published values (`ba_n = -0.00553`, `ba_d = 0.090165`, `ba_i = 141.50225`) do not apply, which is why this clock gets its own `mortality_to_phenoage_saopaulo` postprocess rather than reusing `mortality_to_phenoage`.

In [8]:
base_model = pya.models.LinearModel(input_dim=len(model.features))
base_model.linear.weight.data = torch.tensor([params["coefficients"]], dtype=torch.float64)
base_model.linear.bias.data = torch.tensor([params["intercept"]], dtype=torch.float64)
model.base_model = base_model

for name in ["m_n", "m_d", "ba_n", "ba_d", "ba_i"]:
    setattr(model, name, torch.tensor(params[name], dtype=torch.float64))

pd.DataFrame(
    {"constant": ["m_n", "m_d", "ba_n", "ba_d", "ba_i"],
     "refit": [params[name] for name in ["m_n", "m_d", "ba_n", "ba_d", "ba_i"]],
     "levine_published": [-1.51714, 0.007692696, -0.0055305, 0.090165, 141.50225]}
)

,constant,refit,levine_published
0,m_n,-1.357302,-1.517140
1,m_d,0.007146,0.007693
2,ba_n,-0.005808,-0.005530
3,ba_d,0.087422,0.090165
4,ba_i,141.825450,141.502250


## Load reference values

`check_features_in_adata` substitutes these for any feature a user's dataframe does not carry. The pipeline's own fallback is `0`, which is not a physiological value for any of these assays and would push the linear predictor far off.

Because the clock is linear in its features, there is an exactly-right answer for a single substituted value: the predictor's **training mean**. Substituting it makes the absent term contribute the population-average amount to the linear predictor, which is the least-wrong single constant available — it is the value that makes the substitution unbiased over the training sample.

`extract_bioage_params.R` takes those means over the `model.frame` of the gompertz fit rather than over the filtered data frame, because `flexsurvreg` drops incomplete cases; the training sample is the rows the fit actually saw. They are stored on the fitted scale, so the CRP entry is `log1p(CRP in mg/dL)` and has to be inverted back to raw mg/dL here — `preprocess` will apply `log1p` to whatever sits in that slot.

In [9]:
crp = model.features.index("c_reactive_protein")
reference = list(params["training_mean"])
reference[crp] = math.expm1(reference[crp])  # stored raw; preprocess applies log1p

model.reference_values = reference

assert len(model.reference_values) == len(model.features)
pd.DataFrame({"feature": model.features, "reference_value": model.reference_values})

,feature,reference_value
0,glucose,5.447648
1,c_reactive_protein,0.376398
2,lymphocyte_percent,33.035487
3,mean_cell_volume,89.421693
4,red_cell_distribution_width,13.149559
5,white_blood_cell_count,7.171985
6,age,47.073023


## Load preprocess and postprocess objects

In [10]:
model.preprocess_name = "log1p_crp"
model.preprocess_dependencies = None

In [11]:
model.postprocess_name = "mortality_to_phenoage_saopaulo"
model.postprocess_dependencies = None

## Check all clock parameters

In [12]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'Levine, M. E., et al. "An epigenetic biomarker of aging for '
             'lifespan and healthspan." Aging 10.4 (2018): 573-591.',
 'clock_name': 'phenoagesaopaulo',
 'data_type': 'clinical biomarkers',
 'doi': 'https://doi.org/10.18632/aging.101414',
 'journal': 'Aging',
 'last_author': 'Steve Horvath',
 'model_type': 'Gompertz hazards regression with age calibration',
 'n_features': 7,
 'notes': 'PhenoAge refit on NHANES III adults aged 20-84 with '
          'BioAge::phenoage_calc(), dropping creatinine, albumin, and alkaline '
          'phosphatase and keeping the remaining six biomarkers plus '
          'chronological age. The coefficients and the mortality-to-age '
          'constants are refit, so they differ from the published PhenoAge and '
          "its constants do not apply. Biomarkers are on Bi

## Basic test

The registry bounds are *plausibility* bounds, not reference ranges, so their midpoints are pathological values (creatinine 1505 umol/L, white blood cell count 250). The mortality-to-age link saturates on inputs like these and the prediction overflows to `inf`. That is expected here: this cell is a smoke test that the clock runs end to end, not a check that it predicts sensibly.

The smoke test feeds the midpoint of each feature's registered range, so the inputs are physiologically plausible rather than random.

In [13]:
records = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
midpoints = torch.tensor(
    [[(record["low"] + record["high"]) / 2 for record in records]], dtype=torch.float64
)
model.eval()
model.to(torch.float64)
pred = model(midpoints)
pred

tensor([[inf]], dtype=torch.float64, grad_fn=<AddBackward0>)

#### Parity with BioAge

The acceptance gate is `tests/predict/test_bioage_clocks.py`, which reproduces BioAge's own output for 20 NHANES IV subjects. Reproduced inline here as well, since a notebook that builds parameters should show that they land where the source package lands.

In [14]:
with open("../bioage_params/reference_predictions.json") as handle:
    reference_predictions = json.load(handle)

matrix = torch.tensor(
    [[row[name] for name in model.features] for row in reference_predictions["rows"]], dtype=torch.float64
)
with torch.inference_mode():
    predicted = model(matrix).squeeze(-1)

expected = torch.tensor(reference_predictions["expected"]["phenoagesaopaulo"], dtype=torch.float64)
print("max absolute difference:", (predicted - expected).abs().max().item())

max absolute difference: 4.973799150320701e-12


## Save torch model

In [15]:
torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")

## Clear directory
<a id="10"></a>

In [16]:
# Function to remove a folder and all its contents
def remove_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    except Exception as e:
        print(f"Error deleting folder {path}: {e}")

# Get a list of all files and folders in the current directory
all_items = os.listdir('.')

# Loop through the items
for item in all_items:
    # Check if it's a file and does not end with .ipynb
    if os.path.isfile(item) and not item.endswith('.ipynb'):
        os.remove(item)
        print(f"Deleted file: {item}")
    # Check if it's a folder
    elif os.path.isdir(item):
        remove_folder(item)